In [73]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [106]:
# Read training data
train_df = pd.read_csv('../data/train.csv')
train_df

,match_id,color,rank,map_code,duration,car_name,possession_time,time_in_side,shots,shots_against,...,percent_defensive_half,percent_offensive_half,percent_behind_ball,percent_infront_ball,percent_most_back,percent_most_forward,percent_closest_to_ball,percent_farthest_from_ball,demos_inflicted,demos_taken
0,0,blue,silver,neotokyo_standard_p,163,Breakout,54.95,58.96,2,4,...,62.736664,37.263340,66.842890,33.157120,95.534080,95.534080,95.534080,95.534080,1,0
1,0,orange,silver,neotokyo_standard_p,163,Octane,27.51,80.68,4,2,...,63.576576,36.423428,77.846670,22.153326,98.304170,98.304170,98.304170,98.304170,0,1
2,1,blue,gold,stadium_foggy_p,460,Octane,132.98,244.72,7,10,...,75.199120,24.800879,73.992320,26.007679,96.942440,96.942440,96.942440,96.942440,0,0
3,1,orange,gold,stadium_foggy_p,460,Octane,102.55,148.84,10,7,...,55.832005,44.167990,77.792800,22.207201,96.918510,96.918510,96.918510,96.918510,0,0
4,2,blue,silver,neotokyo_standard_p,94,Octane,25.24,33.70,0,3,...,76.376495,23.623503,68.454840,31.545156,93.636470,93.636470,93.636470,93.636470,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60237,30118,orange,silver,chn_stadium_p,390,Ronin GXT,94.20,193.21,8,6,...,64.023270,35.976734,72.403404,27.596594,98.048460,98.048460,98.048460,98.048460,0,1
60238,30119,blue,silver,trainstation_p,372,Fennec,145.58,123.44,8,3,...,56.082024,43.917976,78.380820,21.619183,98.296730,98.296730,98.296730,98.296730,0,2
60239,30119,orange,silver,trainstation_p,372,Octane,119.14,213.29,3,8,...,72.444016,27.555986,70.517235,29.482769,100.418655,100.418655,100.418655,100.418655,2,0
60240,30120,blue,platinum,wasteland_night_s_p,425,Octane,131.41,184.57,7,7,...,58.544390,41.455605,73.541850,26.458149,97.486570,97.486570,97.486570,97.486570,1,0


In [59]:
# Training data column info
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60242 entries, 0 to 60241
Data columns (total 91 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   match_id                            60242 non-null  int64  
 1   color                               60242 non-null  object 
 2   rank                                60242 non-null  object 
 3   map_code                            60242 non-null  object 
 4   duration                            60242 non-null  int64  
 5   car_name                            60242 non-null  object 
 6   possession_time                     60198 non-null  float64
 7   time_in_side                        60228 non-null  float64
 8   shots                               60242 non-null  int64  
 9   shots_against                       60242 non-null  int64  
 10  goals                               60242 non-null  int64  
 11  goals_against                       60242

In [88]:
# Get columns with NaN values
train_df.columns[train_df.isna().any()]

Index(['possession_time', 'time_in_side', 'goals_against_while_last_defender'], dtype='object')

In [110]:
# Create dataset
match_df = train_df.groupby(['match_id', 'rank']).sum().reset_index()
match_df

,match_id,rank,color,map_code,duration,car_name,possession_time,time_in_side,shots,shots_against,...,percent_defensive_half,percent_offensive_half,percent_behind_ball,percent_infront_ball,percent_most_back,percent_most_forward,percent_closest_to_ball,percent_farthest_from_ball,demos_inflicted,demos_taken
0,0,silver,blueorange,neotokyo_standard_pneotokyo_standard_p,326,BreakoutOctane,82.46,139.64,6,6,...,126.313240,73.686768,144.689560,55.310446,193.838250,193.838250,193.838250,193.838250,1,1
1,1,gold,blueorange,stadium_foggy_pstadium_foggy_p,920,OctaneOctane,235.53,393.56,17,17,...,131.031125,68.968869,151.785120,48.214880,193.860950,193.860950,193.860950,193.860950,0,0
2,2,silver,blueorange,neotokyo_standard_pneotokyo_standard_p,188,OctaneOctane,43.25,81.11,3,3,...,131.124745,68.875258,144.769240,55.230766,187.390130,187.390130,187.390130,187.390130,0,0
3,3,platinum,blueorange,stadium_day_pstadium_day_p,686,FennecOctane,229.37,301.88,12,12,...,125.121784,74.878216,152.124215,47.875778,195.873688,195.873688,195.873688,195.873688,2,2
4,4,platinum,blueorange,trainstation_dawn_ptrainstation_dawn_p,732,OctaneOctane,260.79,330.97,16,16,...,120.835720,79.164277,151.986890,48.013109,196.647236,196.647236,196.647236,196.647236,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30116,30116,platinum,blueorange,neotokyo_standard_pneotokyo_standard_p,656,OctaneOctane,206.86,284.36,13,13,...,120.551075,79.448913,144.377590,55.622410,193.610440,193.610440,193.610440,193.610440,0,0
30117,30117,bronze,blueorange,neotokyo_standard_pneotokyo_standard_p,852,OctaneOctane,211.08,365.22,20,20,...,128.013235,71.986763,155.357760,44.642239,195.385700,195.385700,195.385700,195.385700,1,1
30118,30118,silver,blueorange,chn_stadium_pchn_stadium_p,780,Mudcat GXTRonin GXT,249.92,344.39,14,14,...,119.381703,80.618304,142.307998,57.692005,196.069200,196.069200,196.069200,196.069200,1,1
30119,30119,silver,blueorange,trainstation_ptrainstation_p,744,FennecOctane,264.72,336.73,11,11,...,128.526040,71.473962,148.898055,51.101952,198.715385,198.715385,198.715385,198.715385,2,2


In [118]:
# Set features and target
X = match_df.drop(columns=['match_id', 'rank', 'color', 'car_name'])
y = match_df['rank']

In [119]:
# Split train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [120]:
# Initialize column transformer
ct = ColumnTransformer(
    transformers=[
        ('ohe', OneHotEncoder(), ['map_code']),
        ('imputer', SimpleImputer(), ['possession_time', 'time_in_side', 'goals_against_while_last_defender'])
    ],
    remainder='passthrough'
)

In [121]:
# Create and fit model pipeline
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('scaler', StandardScaler()),
        ('model', MLPClassifier(hidden_layer_sizes=(2,)))
    ]
).fit(X_train, y_train)

In [122]:
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

In [123]:
confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
)

array([[ 224,    1,    2,   52,   10,  227],
       [   6, 2964, 1007,    9,  122,    7],
       [  16, 1110, 2382,   99, 1208,   26],
       [  57,    4,  117, 2638, 1165,  395],
       [  32,  102, 1120, 1109, 2816,   69],
       [ 130,    0,    8,  910,   57,  883]])

In [124]:
confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)

array([[  81,    0,    0,   25,    9,  106],
       [   1, 1246,  449,    7,   59,    1],
       [   4,  510,  975,   48,  533,    5],
       [  20,    2,   51, 1121,  514,  168],
       [   9,   40,  491,  487, 1192,   31],
       [  58,    1,    2,  413,   29,  349]])

In [125]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))
print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

              precision    recall  f1-score   support

      bronze       0.48      0.43      0.46       516
    champion       0.71      0.72      0.71      4115
     diamond       0.51      0.49      0.50      4841
        gold       0.55      0.60      0.57      4376
    platinum       0.52      0.54      0.53      5248
      silver       0.55      0.44      0.49      1988

    accuracy                           0.56     21084
   macro avg       0.55      0.54      0.54     21084
weighted avg       0.56      0.56      0.56     21084

              precision    recall  f1-score   support

      bronze       0.47      0.37      0.41       221
    champion       0.69      0.71      0.70      1763
     diamond       0.50      0.47      0.48      2075
        gold       0.53      0.60      0.56      1876
    platinum       0.51      0.53      0.52      2250
      silver       0.53      0.41      0.46       852

    accuracy                           0.55      9037
   macro avg       0.54

In [96]:
test_df = pd.read_csv('../data/test.csv')

In [126]:
match_test_df = test_df.groupby(['match_id']).sum().reset_index()

In [127]:
y_pred = pipe.predict(match_test_df.drop(columns=['match_id']))

In [128]:
converter = { 'bronze': 1, 'silver': 2, 'gold': 3, 'platinum': 4, 'diamond': 5, 'champion': 6 }

y_pred = pd.Series(y_pred).map(converter)

In [129]:
submission = pd.concat([match_test_df['match_id'], y_pred], axis = 1).rename(columns = {0: 'rank'})

In [132]:
submission.to_csv('../data/mlpclassification_all_submission.csv', index=False)

In [133]:
submission.shape

(2500, 2)